# EMCAD Experiment 2 — BraTS Run 1
## Part 2: Training Epochs 1–50

This notebook starts **Run 1** of the BraTS-domain experiment and trains only epochs **1–50 / 200**.

The experiment is intentionally divided into four GPU-safe stages:

- Part 2: epochs 1–50
- Part 3: epochs 51–100
- Part 4: epochs 101–150
- Part 5: epochs 151–200 + full held-out test evaluation

At the end of this notebook, a complete resume checkpoint is saved containing the model, optimizer, scheduler, RNG states, DataLoader shuffle-generator state, best-validation state, and training history. The next notebook continues the **same run**, not a new run.

### Kaggle inputs
Add exactly these two saved notebook outputs:

1. **Notebook 1 — Environment Preparation**
2. **EMCAD Experiment 2 — BraTS Preparation** (completed Part 1)

### Fixed training protocol
- Architecture: original `PVTv2-B2 + EMCAD`
- Author repository commit: `26c9c31f731f749b62c5fe83f44376dac75f3aa8`
- Binary target: Whole Tumour
- Input: FLAIR + T1-Gd + T2
- Image size: 352×352
- Batch size: 8
- Optimizer: AdamW
- LR: 0.0005
- Weight decay: 1e-4
- Scheduler: CosineAnnealingLR, `T_max=200`, `eta_min=1e-6`
- Multi-scale training: `[0.75, 1.0, 1.25]`
- Gradient clip: 0.5
- Loss: original EMCAD five-term binary structure loss
- Validation: cached validation slices only
- Model selection: best validation Dice



In [1]:
from pathlib import Path
import hashlib
import shutil
import subprocess
import tarfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def select_unique(filename, label):
    files = sorted(set(INPUT.rglob(filename)))

    if not files:
        raise FileNotFoundError(f"{label} not found: {filename}")

    if len(files) == 1:
        return files[0]

    groups = {}
    for path in files:
        groups.setdefault(sha256(path), []).append(path)

    if len(groups) == 1:
        print(f"{label}: {len(files)} identical copies found; using first.")
        return files[0]

    raise RuntimeError(f"{label}: conflicting copies found: {files}")

env_archive = select_unique(
    "emcadenv.tar.gz",
    "Notebook 1 environment",
)

repo_archive = select_unique(
    "EMCAD_prepared.tar.gz",
    "Notebook 1 EMCAD repository",
)

notebook1_checksums = select_unique(
    "EMCAD_NOTEBOOK1_SHA256SUMS.txt",
    "Notebook 1 checksums",
)

brats_archive = select_unique(
    "EMCAD_BraTS_Part1_Output.tar.gz",
    "BraTS Part 1 prepared archive",
)

brats_checksums = select_unique(
    "EMCAD_BraTS_Part1_SHA256SUMS.txt",
    "BraTS Part 1 checksums",
)

print("Notebook 1:", env_archive.parent)
print("BraTS Part 1:", brats_archive.parent)


Notebook 1: /kaggle/input/notebooks/anirbanroysudipto/notebook-1-environment-preparation
BraTS Part 1: /kaggle/input/notebooks/rinkyranisaha/emcad-experiment-2-brats-preparation


In [2]:
# Verify the two large input archives against their saved manifests.

def expected_hash(manifest, filename):
    for line in manifest.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 2 and parts[-1] == filename:
            return parts[0]
    raise RuntimeError(
        f"{filename} not found in checksum manifest {manifest}"
    )

expected_env = expected_hash(
    notebook1_checksums,
    env_archive.name,
)
expected_repo = expected_hash(
    notebook1_checksums,
    repo_archive.name,
)
expected_brats = expected_hash(
    brats_checksums,
    brats_archive.name,
)

assert sha256(env_archive) == expected_env
assert sha256(repo_archive) == expected_repo
assert sha256(brats_archive) == expected_brats

print("Notebook 1 environment checksum: PASS")
print("Notebook 1 repository checksum: PASS")
print("BraTS Part 1 archive checksum: PASS")


Notebook 1 environment checksum: PASS
Notebook 1 repository checksum: PASS
BraTS Part 1 archive checksum: PASS


In [3]:
ENV = WORK / "emcadenv"
REPO = WORK / "EMCAD"
BRATS = WORK / "EMCAD_BraTS_Part1_Output"

for path in [ENV, REPO, BRATS]:
    if path.exists():
        shutil.rmtree(path)

ENV.mkdir()

with tarfile.open(env_archive, "r:gz") as tf:
    tf.extractall(ENV)

subprocess.run(
    [str(ENV / "bin" / "conda-unpack")],
    check=True,
)

with tarfile.open(repo_archive, "r:gz") as tf:
    tf.extractall(WORK)

with tarfile.open(brats_archive, "r:gz") as tf:
    tf.extractall(WORK)

assert REPO.is_dir()
assert BRATS.is_dir()

EXPECTED_COMMIT = "26c9c31f731f749b62c5fe83f44376dac75f3aa8"

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

assert commit == EXPECTED_COMMIT

subprocess.run(
    ["git", "-C", str(REPO), "diff", "--exit-code"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO), "diff", "--cached", "--exit-code"],
    check=True,
)

print("Author commit:", commit)
print("Tracked author source: CLEAN")
print("BraTS prepared data restored:", BRATS)


/tmp/ipykernel_23/1979387457.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(ENV)
/tmp/ipykernel_23/1979387457.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(WORK)
/tmp/ipykernel_23/1979387457.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(WORK)


Author commit: 26c9c31f731f749b62c5fe83f44376dac75f3aa8
Tracked author source: CLEAN
BraTS prepared data restored: /kaggle/working/EMCAD_BraTS_Part1_Output


In [4]:
# Verify the internal BraTS preparation artifacts after extraction.

for filename in [
    "patient_split.json",
    "patient_audit.csv",
    "slice_cache_manifest.csv",
    "test_volume_manifest.csv",
    "preparation_metadata.json",
]:
    expected = expected_hash(brats_checksums, filename)
    path = BRATS / filename

    assert path.is_file(), path
    assert sha256(path) == expected, filename

train_cache = sorted((BRATS / "slice_cache" / "train").glob("*.pt"))
val_cache = sorted((BRATS / "slice_cache" / "val").glob("*.pt"))
test_images = sorted((BRATS / "test_volumes" / "images").glob("*.nii.gz"))
test_labels = sorted((BRATS / "test_volumes" / "labels").glob("*.nii.gz"))

assert len(train_cache) == 388
assert len(val_cache) == 48
assert len(test_images) == 48
assert len(test_labels) == 48

print("Train patient caches:", len(train_cache))
print("Validation patient caches:", len(val_cache))
print("Held-out test image volumes:", len(test_images))
print("Held-out test label volumes:", len(test_labels))
print("Prepared-data integrity: PASS")


Train patient caches: 388
Validation patient caches: 48
Held-out test image volumes: 48
Held-out test label volumes: 48
Prepared-data integrity: PASS


In [5]:
subprocess.run(
    [
        str(ENV / "bin" / "python"),
        "-c",
        "import sys, torch, torchvision, timm; "
        "print('Python:', sys.version.split()[0]); "
        "print('PyTorch:', torch.__version__); "
        "print('Torchvision:', torchvision.__version__); "
        "print('timm:', timm.__version__); "
        "print('CUDA available:', torch.cuda.is_available()); "
        "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'); "
        "assert sys.version_info[:2] == (3, 8); "
        "assert torch.__version__.startswith('1.11.0'); "
        "assert torch.cuda.is_available()"
    ],
    check=True,
)


Python: 3.8.20
PyTorch: 1.11.0+cu113
Torchvision: 0.12.0+cu113
timm: 0.6.12
CUDA available: True
GPU: Tesla T4


CompletedProcess(args=['/kaggle/working/emcadenv/bin/python', '-c', "import sys, torch, torchvision, timm; print('Python:', sys.version.split()[0]); print('PyTorch:', torch.__version__); print('Torchvision:', torchvision.__version__); print('timm:', timm.__version__); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'); assert sys.version_info[:2] == (3, 8); assert torch.__version__.startswith('1.11.0'); assert torch.cuda.is_available()"], returncode=0)

In [6]:
from pathlib import Path

training_code = '\nimport csv\nimport json\nimport logging\nimport math\nimport random\nimport time\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.optim.lr_scheduler import CosineAnnealingLR\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.transforms import InterpolationMode\nfrom torchvision.transforms import functional as TF\n\nfrom lib.networks import EMCADNet\nfrom train_polyp import structure_loss\nfrom utils.utils import adjust_lr, clip_gradient\n\n\n# =============================================================================\n# Configuration\n# =============================================================================\n\nWORK = Path("/kaggle/working")\nDATA_ROOT = WORK / "EMCAD_BraTS_Part1_Output"\nOUTPUT_ROOT = WORK / "EMCAD_BraTS_Run1_Part2"\n\nSEED = 2026\n\nSTART_EPOCH = 1\nEND_EPOCH = 50\nTOTAL_EPOCHS = 200\n\nIMG_SIZE = 352\nBATCH_SIZE = 8\nVAL_BATCH_SIZE = 16\n\nLR = 5e-4\nWEIGHT_DECAY = 1e-4\nETA_MIN = 1e-6\n\nCLIP = 0.5\nDECAY_RATE = 0.1\nDECAY_EPOCH = 300\n\nSIZE_RATES = [0.75, 1.0, 1.25]\nTHRESHOLD = 0.5\n\nKERNEL_SIZES = [1, 3, 5]\nEXPANSION_FACTOR = 2\nLGAG_KS = 3\nACTIVATION = "relu6"\nENCODER = "pvt_v2_b2"\nPRETRAINED_DIR = "./pretrained_pth/pvt/"\n\nAUTHOR_COMMIT = "26c9c31f731f749b62c5fe83f44376dac75f3aa8"\n\nIMAGENET_MEAN = torch.tensor(\n    [0.485, 0.456, 0.406]\n).view(3, 1, 1)\n\nIMAGENET_STD = torch.tensor(\n    [0.229, 0.224, 0.225]\n).view(3, 1, 1)\n\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n\n\n# =============================================================================\n# Reproducibility\n# =============================================================================\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\n# =============================================================================\n# Prepared-data preflight\n# =============================================================================\n\ndef load_preparation_metadata():\n    split = json.loads(\n        (DATA_ROOT / "patient_split.json").read_text()\n    )\n\n    metadata = json.loads(\n        (DATA_ROOT / "preparation_metadata.json").read_text()\n    )\n\n    assert len(split["train"]) == 388\n    assert len(split["val"]) == 48\n    assert len(split["test"]) == 48\n\n    assert not (set(split["train"]) & set(split["val"]))\n    assert not (set(split["train"]) & set(split["test"]))\n    assert not (set(split["val"]) & set(split["test"]))\n\n    assert metadata["target"] == "Whole Tumour (WT): label > 0"\n\n    print("Patient split: 388 / 48 / 48")\n    print("Target:", metadata["target"])\n    print(\n        "Input modalities:",\n        metadata["selected_input_modalities"],\n    )\n    print("Patient leakage check: PASS")\n\n    return split, metadata\n\n\n# =============================================================================\n# PyTorch datasets\n# =============================================================================\n\ndef preprocess(image, mask, training):\n    image = image.float()\n    mask = mask.float()\n\n    if training:\n        if random.random() < 0.5:\n            angle = random.uniform(-90.0, 90.0)\n\n            image = TF.rotate(\n                image,\n                angle,\n                interpolation=InterpolationMode.BILINEAR,\n            )\n\n            mask = TF.rotate(\n                mask,\n                angle,\n                interpolation=InterpolationMode.NEAREST,\n            )\n\n        if random.random() < 0.5:\n            image = TF.hflip(image)\n            mask = TF.hflip(mask)\n\n        if random.random() < 0.5:\n            image = TF.vflip(image)\n            mask = TF.vflip(mask)\n\n    image = TF.resize(\n        image,\n        [IMG_SIZE, IMG_SIZE],\n        interpolation=InterpolationMode.BILINEAR,\n    )\n\n    mask = TF.resize(\n        mask,\n        [IMG_SIZE, IMG_SIZE],\n        interpolation=InterpolationMode.NEAREST,\n    )\n\n    image = (\n        image - IMAGENET_MEAN\n    ) / IMAGENET_STD\n\n    mask = (mask > 0.5).float()\n\n    return image, mask\n\n\nclass TrainPatientDataset(Dataset):\n    """\n    One training item = one patient.\n\n    Each epoch draws one slice at random from that patient\'s prepared\n    2D slice bank. This keeps patient representation balanced while\n    presenting different tumour/non-tumour slices across epochs.\n    """\n\n    def __init__(self):\n        paths = sorted(\n            (DATA_ROOT / "slice_cache" / "train").glob("*.pt")\n        )\n\n        if len(paths) != 388:\n            raise RuntimeError(\n                f"Expected 388 train patient caches, found {len(paths)}."\n            )\n\n        self.cases = [\n            torch.load(path, map_location="cpu")\n            for path in paths\n        ]\n\n    def __len__(self):\n        return len(self.cases)\n\n    def __getitem__(self, index):\n        case = self.cases[index]\n\n        sample_index = random.randrange(\n            case["images"].shape[0]\n        )\n\n        image = case["images"][sample_index]\n        mask = case["masks"][sample_index]\n\n        return preprocess(\n            image,\n            mask,\n            training=True,\n        )\n\n\nclass ValidationSliceDataset(Dataset):\n    """\n    Validation is deterministic: every cached validation slice is used.\n    """\n\n    def __init__(self):\n        paths = sorted(\n            (DATA_ROOT / "slice_cache" / "val").glob("*.pt")\n        )\n\n        if len(paths) != 48:\n            raise RuntimeError(\n                f"Expected 48 validation patient caches, found {len(paths)}."\n            )\n\n        self.samples = []\n\n        for path in paths:\n            case = torch.load(\n                path,\n                map_location="cpu",\n            )\n\n            for index in range(\n                case["images"].shape[0]\n            ):\n                self.samples.append(\n                    (\n                        case["patient_id"],\n                        int(\n                            case["slice_indices"][index].item()\n                        ),\n                        case["images"][index],\n                        case["masks"][index],\n                    )\n                )\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, index):\n        patient_id, slice_index, image, mask = (\n            self.samples[index]\n        )\n\n        image, mask = preprocess(\n            image,\n            mask,\n            training=False,\n        )\n\n        return (\n            image,\n            mask,\n            patient_id,\n            slice_index,\n        )\n\n\ndef build_loaders():\n    train_dataset = TrainPatientDataset()\n    val_dataset = ValidationSliceDataset()\n\n    shuffle_generator = torch.Generator()\n    shuffle_generator.manual_seed(SEED)\n\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=BATCH_SIZE,\n        shuffle=True,\n        num_workers=0,\n        pin_memory=True,\n        generator=shuffle_generator,\n    )\n\n    val_loader = DataLoader(\n        val_dataset,\n        batch_size=VAL_BATCH_SIZE,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=True,\n    )\n\n    print("Train patients per epoch:", len(train_dataset))\n    print("Train batches per epoch:", len(train_loader))\n    print("Validation slices:", len(val_dataset))\n    print("Validation batches:", len(val_loader))\n\n    return train_loader, val_loader, shuffle_generator\n\n\n# =============================================================================\n# Model and author-style loss\n# =============================================================================\n\ndef build_model():\n    model = EMCADNet(\n        num_classes=1,\n        kernel_sizes=KERNEL_SIZES,\n        expansion_factor=EXPANSION_FACTOR,\n        dw_parallel=True,\n        add=True,\n        lgag_ks=LGAG_KS,\n        activation=ACTIVATION,\n        encoder=ENCODER,\n        pretrain=True,\n        pretrained_dir=PRETRAINED_DIR,\n    )\n\n    return model\n\n\ndef emcad_loss(outputs, masks):\n    if not isinstance(outputs, list):\n        outputs = [outputs]\n\n    if len(outputs) != 4:\n        raise RuntimeError(\n            f"Expected four EMCAD outputs, got {len(outputs)}."\n        )\n\n    loss_p1 = structure_loss(outputs[0], masks)\n    loss_p2 = structure_loss(outputs[1], masks)\n    loss_p3 = structure_loss(outputs[2], masks)\n    loss_p4 = structure_loss(outputs[3], masks)\n\n    loss_p1234 = structure_loss(\n        outputs[0]\n        + outputs[1]\n        + outputs[2]\n        + outputs[3],\n        masks,\n    )\n\n    return (\n        loss_p1\n        + loss_p2\n        + loss_p3\n        + loss_p4\n        + loss_p1234\n    )\n\n\n# =============================================================================\n# Metrics\n# =============================================================================\n\ndef confusion_counts(prediction, target):\n    prediction = prediction.bool()\n    target = target.bool()\n\n    tp = torch.logical_and(\n        prediction,\n        target,\n    ).sum().item()\n\n    tn = torch.logical_and(\n        ~prediction,\n        ~target,\n    ).sum().item()\n\n    fp = torch.logical_and(\n        prediction,\n        ~target,\n    ).sum().item()\n\n    fn = torch.logical_and(\n        ~prediction,\n        target,\n    ).sum().item()\n\n    return tp, tn, fp, fn\n\n\ndef metrics_from_counts(tp, tn, fp, fn):\n    eps = 1e-6\n\n    return {\n        "dice": (\n            2 * tp + eps\n        ) / (\n            2 * tp + fp + fn + eps\n        ),\n        "iou": (\n            tp + eps\n        ) / (\n            tp + fp + fn + eps\n        ),\n        "sensitivity": (\n            tp + eps\n        ) / (\n            tp + fn + eps\n        ),\n        "specificity": (\n            tn + eps\n        ) / (\n            tn + fp + eps\n        ),\n        "precision": (\n            tp + eps\n        ) / (\n            tp + fp + eps\n        ),\n        "accuracy": (\n            tp + tn + eps\n        ) / (\n            tp + tn + fp + fn + eps\n        ),\n    }\n\n\ndef validate(model, loader, device):\n    model.eval()\n\n    tp = tn = fp = fn = 0\n\n    with torch.no_grad():\n        for images, masks, _, _ in loader:\n            images = images.to(\n                device,\n                non_blocking=True,\n            )\n\n            masks = masks.to(\n                device,\n                non_blocking=True,\n            ).float()\n\n            outputs = model(images)\n\n            if not isinstance(outputs, list):\n                outputs = [outputs]\n\n            logits = outputs[-1]\n\n            predictions = (\n                torch.sigmoid(logits)\n                >= THRESHOLD\n            )\n\n            targets = masks >= 0.5\n\n            counts = confusion_counts(\n                predictions,\n                targets,\n            )\n\n            tp += counts[0]\n            tn += counts[1]\n            fp += counts[2]\n            fn += counts[3]\n\n    return metrics_from_counts(\n        tp,\n        tn,\n        fp,\n        fn,\n    )\n\n\n# =============================================================================\n# Training\n# =============================================================================\n\ndef train_one_epoch(\n    model,\n    loader,\n    optimizer,\n    epoch,\n    device,\n):\n    model.train()\n\n    running_loss = 0.0\n    samples = 0\n\n    for step, (images, masks) in enumerate(\n        loader,\n        start=1,\n    ):\n        images = images.to(\n            device,\n            non_blocking=True,\n        )\n\n        masks = masks.to(\n            device,\n            non_blocking=True,\n        ).float()\n\n        for rate in SIZE_RATES:\n            optimizer.zero_grad()\n\n            if rate != 1.0:\n                train_size = int(\n                    round(\n                        IMG_SIZE * rate / 32\n                    ) * 32\n                )\n\n                current_images = F.interpolate(\n                    images,\n                    size=(train_size, train_size),\n                    mode="bilinear",\n                    align_corners=True,\n                )\n\n                current_masks = F.interpolate(\n                    masks,\n                    size=(train_size, train_size),\n                    mode="nearest",\n                )\n\n            else:\n                current_images = images\n                current_masks = masks\n\n            outputs = model(current_images)\n\n            loss = emcad_loss(\n                outputs,\n                current_masks,\n            )\n\n            loss.backward()\n\n            clip_gradient(\n                optimizer,\n                CLIP,\n            )\n\n            optimizer.step()\n\n            if rate == 1.0:\n                batch_size = images.size(0)\n\n                running_loss += (\n                    loss.item() * batch_size\n                )\n\n                samples += batch_size\n\n        if (\n            step % 25 == 0\n            or step == len(loader)\n        ):\n            print(\n                f"{datetime.now()} "\n                f"Epoch [{epoch:03d}/{TOTAL_EPOCHS:03d}], "\n                f"Step [{step:04d}/{len(loader):04d}], "\n                f"LR: {optimizer.param_groups[0][\'lr\']:.6f}, "\n                f"Loss: {running_loss / samples:.4f}"\n            )\n\n    return running_loss / samples\n\n\ndef save_history(path, history):\n    if not history:\n        return\n\n    with path.open("w", newline="") as f:\n        writer = csv.DictWriter(\n            f,\n            fieldnames=history[0].keys(),\n        )\n\n        writer.writeheader()\n        writer.writerows(history)\n\n\ndef save_resume_state(\n    path,\n    epoch,\n    model,\n    optimizer,\n    scheduler,\n    shuffle_generator,\n    best_val_dice,\n    best_epoch,\n    history,\n):\n    state = {\n        "stage": "BraTS Run 1 Part 2",\n        "epoch": epoch,\n        "total_epochs": TOTAL_EPOCHS,\n        "seed": SEED,\n        "author_commit": AUTHOR_COMMIT,\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "scheduler_state_dict": scheduler.state_dict(),\n        "best_val_dice": best_val_dice,\n        "best_epoch": best_epoch,\n        "history": history,\n        "python_random_state": random.getstate(),\n        "numpy_random_state": np.random.get_state(),\n        "torch_rng_state": torch.get_rng_state(),\n        "cuda_rng_state_all": torch.cuda.get_rng_state_all(),\n        "shuffle_generator_state": shuffle_generator.get_state(),\n        "config": {\n            "img_size": IMG_SIZE,\n            "batch_size": BATCH_SIZE,\n            "lr": LR,\n            "weight_decay": WEIGHT_DECAY,\n            "eta_min": ETA_MIN,\n            "clip": CLIP,\n            "size_rates": SIZE_RATES,\n            "kernel_sizes": KERNEL_SIZES,\n            "expansion_factor": EXPANSION_FACTOR,\n            "lgag_ks": LGAG_KS,\n            "activation": ACTIVATION,\n            "encoder": ENCODER,\n        },\n    }\n\n    torch.save(\n        state,\n        path,\n    )\n\n\n# =============================================================================\n# Main\n# =============================================================================\n\ndef main():\n    set_seed(SEED)\n\n    split, preparation_metadata = (\n        load_preparation_metadata()\n    )\n\n    device = torch.device(\n        "cuda"\n        if torch.cuda.is_available()\n        else "cpu"\n    )\n\n    if device.type != "cuda":\n        raise RuntimeError(\n            "GPU is required for training."\n        )\n\n    print("Device:", device)\n    print("GPU:", torch.cuda.get_device_name(0))\n    print("PyTorch:", torch.__version__)\n\n    train_loader, val_loader, shuffle_generator = (\n        build_loaders()\n    )\n\n    # Cache compatibility preflight before model training.\n    images, masks = next(\n        iter(\n            DataLoader(\n                train_loader.dataset,\n                batch_size=2,\n                shuffle=False,\n                num_workers=0,\n            )\n        )\n    )\n\n    print("Preflight image batch:", tuple(images.shape))\n    print("Preflight mask batch:", tuple(masks.shape))\n    print(\n        "Preflight foreground fraction:",\n        f"{masks.mean().item():.6f}",\n    )\n\n    assert images.shape == (\n        2,\n        3,\n        IMG_SIZE,\n        IMG_SIZE,\n    )\n\n    assert masks.shape == (\n        2,\n        1,\n        IMG_SIZE,\n        IMG_SIZE,\n    )\n\n    assert masks.max().item() == 1.0\n\n    # Reset all RNG after the preflight so it cannot alter Run 1.\n    set_seed(SEED)\n    shuffle_generator.manual_seed(SEED)\n\n    model = build_model().to(device)\n\n    parameter_count = sum(\n        parameter.numel()\n        for parameter in model.parameters()\n    )\n\n    print(\n        f"Encoder: {ENCODER} | Decoder: EMCAD"\n    )\n\n    print(\n        "Total parameters:",\n        f"{parameter_count / 1e6:.3f}M",\n    )\n\n    # Model preflight.\n    model.eval()\n\n    with torch.no_grad():\n        x = torch.randn(\n            1,\n            3,\n            IMG_SIZE,\n            IMG_SIZE,\n            device=device,\n        )\n\n        outputs = model(x)\n\n    assert isinstance(outputs, list)\n    assert len(outputs) == 4\n\n    for index, output in enumerate(\n        outputs,\n        start=1,\n    ):\n        print(\n            f"Output p{index}:",\n            tuple(output.shape),\n        )\n\n        assert output.shape == (\n            1,\n            1,\n            IMG_SIZE,\n            IMG_SIZE,\n        )\n\n    del x, outputs\n    torch.cuda.empty_cache()\n\n    # Reset again after model-forward preflight.\n    set_seed(SEED)\n    shuffle_generator.manual_seed(SEED)\n\n    # Rebuild the model so the experimental run begins from a clean,\n    # deterministic initialization and loads the original pretrained encoder.\n    del model\n    torch.cuda.empty_cache()\n\n    model = build_model().to(device)\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        LR,\n        weight_decay=WEIGHT_DECAY,\n    )\n\n    scheduler = CosineAnnealingLR(\n        optimizer,\n        T_max=TOTAL_EPOCHS,\n        eta_min=ETA_MIN,\n    )\n\n    log_path = OUTPUT_ROOT / "training_part2.log"\n\n    logging.basicConfig(\n        filename=log_path,\n        level=logging.INFO,\n        format="[%(asctime)s] %(message)s",\n        force=True,\n    )\n\n    best_val_dice = -1.0\n    best_epoch = 0\n    history = []\n\n    best_path = OUTPUT_ROOT / "best_through_epoch050.pth"\n\n    start_time = time.time()\n\n    for epoch in range(\n        START_EPOCH,\n        END_EPOCH + 1,\n    ):\n        epoch_start = time.time()\n\n        # This is kept to mirror the pinned author training loop.\n        # With decay_epoch=300 it is neutral for epochs 1–200.\n        adjust_lr(\n            optimizer,\n            LR,\n            epoch,\n            DECAY_RATE,\n            DECAY_EPOCH,\n        )\n\n        train_loss = train_one_epoch(\n            model=model,\n            loader=train_loader,\n            optimizer=optimizer,\n            epoch=epoch,\n            device=device,\n        )\n\n        val_metrics = validate(\n            model,\n            val_loader,\n            device,\n        )\n\n        epoch_time = time.time() - epoch_start\n\n        row = {\n            "epoch": epoch,\n            "lr": optimizer.param_groups[0]["lr"],\n            "train_loss": train_loss,\n            "val_dice": val_metrics["dice"],\n            "val_iou": val_metrics["iou"],\n            "val_sensitivity": val_metrics["sensitivity"],\n            "val_specificity": val_metrics["specificity"],\n            "val_precision": val_metrics["precision"],\n            "val_accuracy": val_metrics["accuracy"],\n            "epoch_time_seconds": epoch_time,\n        }\n\n        history.append(row)\n\n        message = (\n            f"Epoch: {epoch:03d}, Dataset: val, "\n            f"Dice: {val_metrics[\'dice\']:.4f}, "\n            f"IoU: {val_metrics[\'iou\']:.4f}, "\n            f"Time: {epoch_time:.1f}s"\n        )\n\n        print(message)\n        logging.info(message)\n\n        if val_metrics["dice"] > best_val_dice:\n            previous = best_val_dice\n\n            best_val_dice = val_metrics["dice"]\n            best_epoch = epoch\n\n            torch.save(\n                model.state_dict(),\n                best_path,\n            )\n\n            best_message = (\n                "### Best Model Saved "\n                f"(Dice improved from "\n                f"{previous:.4f} to "\n                f"{best_val_dice:.4f}) ###"\n            )\n\n            print(best_message)\n            logging.info(best_message)\n\n        scheduler.step()\n\n        save_history(\n            OUTPUT_ROOT / "history_through_epoch050.csv",\n            history,\n        )\n\n        # Small rolling safety state. It is overwritten every epoch.\n        save_resume_state(\n            OUTPUT_ROOT / "resume_latest.pth",\n            epoch=epoch,\n            model=model,\n            optimizer=optimizer,\n            scheduler=scheduler,\n            shuffle_generator=shuffle_generator,\n            best_val_dice=best_val_dice,\n            best_epoch=best_epoch,\n            history=history,\n        )\n\n    elapsed = time.time() - start_time\n\n    final_resume = (\n        OUTPUT_ROOT\n        / "resume_epoch050.pth"\n    )\n\n    save_resume_state(\n        final_resume,\n        epoch=END_EPOCH,\n        model=model,\n        optimizer=optimizer,\n        scheduler=scheduler,\n        shuffle_generator=shuffle_generator,\n        best_val_dice=best_val_dice,\n        best_epoch=best_epoch,\n        history=history,\n    )\n\n    metadata = {\n        "experiment": "EMCAD Experiment 2 - BraTS Domain Generalization",\n        "run": 1,\n        "stage": "Part 2",\n        "epochs_completed": [START_EPOCH, END_EPOCH],\n        "next_epoch": END_EPOCH + 1,\n        "total_epochs": TOTAL_EPOCHS,\n        "seed": SEED,\n        "author_commit": AUTHOR_COMMIT,\n        "best_validation_dice_through_epoch_50": best_val_dice,\n        "best_epoch_through_epoch_50": best_epoch,\n        "elapsed_seconds": elapsed,\n        "test_set_evaluated": False,\n        "early_stopping": False,\n    }\n\n    (\n        OUTPUT_ROOT / "part2_metadata.json"\n    ).write_text(\n        json.dumps(\n            metadata,\n            indent=2,\n        )\n    )\n\n    print()\n    print("=" * 72)\n    print("BRATS RUN 1 — PART 2 COMPLETE")\n    print("=" * 72)\n    print("Completed epochs: 1–50 / 200")\n    print("Best validation epoch:", best_epoch)\n    print(\n        "Best validation Dice:",\n        f"{best_val_dice:.4f}",\n    )\n    print(\n        "Stage training time:",\n        f"{elapsed / 3600:.2f} hours",\n    )\n    print("Test set evaluated: NO")\n    print("Next epoch: 51")\n    print("=" * 72)\n\n\nif __name__ == "__main__":\n    main()\n'

script_path = REPO / "experiment2_brats_part2.py"
script_path.write_text(training_code)
print("Training script:", script_path)
print("Lines:", len(training_code.splitlines()))


Training script: /kaggle/working/EMCAD/experiment2_brats_part2.py
Lines: 962


In [7]:
subprocess.run(
    [
        str(ENV / "bin" / "python"),
        "-m",
        "py_compile",
        str(REPO / "experiment2_brats_part2.py"),
    ],
    cwd=REPO,
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO), "diff", "--exit-code"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO), "diff", "--cached", "--exit-code"],
    check=True,
)

print("Python 3.8 syntax check: PASS")
print("Tracked author source remains unchanged.")


Python 3.8 syntax check: PASS
Tracked author source remains unchanged.


## Train epochs 1–50

This is the long-running GPU cell. It does not evaluate the held-out test patients.

A resume state is written after every completed epoch, and the final Part 2 state is saved at epoch 50.


In [8]:
stdout_path = WORK / "EMCAD_BraTS_Run1_Part2_stdout.log"

with stdout_path.open("w") as log:
    process = subprocess.Popen(
        [
            str(ENV / "bin" / "python"),
            "-u",
            "experiment2_brats_part2.py",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Part 2 failed with return code {return_code}. "
        f"See {stdout_path}"
    )

print("Part 2 execution: PASS")


Patient split: 388 / 48 / 48
Target: Whole Tumour (WT): label > 0
Input modalities: {'FLAIR': 0, 'T1-Gd': 2, 'T2': 3}
Patient leakage check: PASS
Device: cuda
GPU: Tesla T4
PyTorch: 1.11.0+cu113
Train patients per epoch: 388
Train batches per epoch: 49
Validation slices: 480
Validation batches: 30
Preflight image batch: (2, 3, 352, 352)
Preflight mask batch: (2, 1, 352, 352)
Preflight foreground fraction: 0.018756
Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515
Encoder: pvt_v2_b2 | Decoder: EMCAD
Total parameters: 26.764M
Output p1: (1, 1, 352, 352)
Output p2: (1, 1, 352, 352)
Output p3: (1, 1, 352, 352)
Output p4: (1, 1, 352, 352)
Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515
2026-10-05 05:15:52.184306 Epoch [001/200], Step [0025/0049], LR: 0.000500, Loss: 4.3042
2026-10-05 05:18:01.318414 Epoch [001/200], Step [0049/0049], LR: 0.000500, Loss: 3.6078
Epoch: 001

## Package the epoch-50 continuation state

Part 3 will need:
- this Part 2 state package,
- the same BraTS Part 1 prepared-data notebook,
- the same Notebook 1 environment preparation.

The test set is still untouched.


In [9]:
import json
import shutil

RESULT = WORK / "EMCAD_BraTS_Run1_Part2"

required = [
    RESULT / "resume_epoch050.pth",
    RESULT / "best_through_epoch050.pth",
    RESULT / "history_through_epoch050.csv",
    RESULT / "part2_metadata.json",
    RESULT / "training_part2.log",
]

for path in required:
    assert path.is_file(), path

metadata = json.loads(
    (RESULT / "part2_metadata.json").read_text()
)

assert metadata["next_epoch"] == 51
assert metadata["test_set_evaluated"] is False

package_root = WORK / "EMCAD_BraTS_Run1_Part2_State"

if package_root.exists():
    shutil.rmtree(package_root)

package_root.mkdir()

for path in required:
    shutil.copy2(
        path,
        package_root / path.name,
    )

shutil.copy2(
    REPO / "experiment2_brats_part2.py",
    package_root / "experiment2_brats_part2.py",
)

shutil.copy2(
    stdout_path,
    package_root / "full_stdout.log",
)

(package_root / "AUTHOR_COMMIT.txt").write_text(
    EXPECTED_COMMIT + "\n"
)

archive = shutil.make_archive(
    str(WORK / "EMCAD_BraTS_Run1_Part2_State"),
    "gztar",
    root_dir=WORK,
    base_dir=package_root.name,
)

checksum_path = WORK / "EMCAD_BraTS_Run1_Part2_SHA256SUMS.txt"

with checksum_path.open("w") as f:
    for path in [
        Path(archive),
        RESULT / "resume_epoch050.pth",
        RESULT / "best_through_epoch050.pth",
        RESULT / "history_through_epoch050.csv",
        RESULT / "part2_metadata.json",
    ]:
        f.write(
            f"{sha256(path)}  {path.name}\n"
        )

print("State package:", archive)
print("State package size:", f"{Path(archive).stat().st_size / 1024**2:.2f} MiB")
print("Checksums:", checksum_path)


State package: /kaggle/working/EMCAD_BraTS_Run1_Part2_State.tar.gz
State package size: 380.07 MiB
Checksums: /kaggle/working/EMCAD_BraTS_Run1_Part2_SHA256SUMS.txt


In [10]:
required_outputs = [
    WORK / "EMCAD_BraTS_Run1_Part2_State.tar.gz",
    WORK / "EMCAD_BraTS_Run1_Part2_SHA256SUMS.txt",
    WORK / "EMCAD_BraTS_Run1_Part2_stdout.log",
]

for path in required_outputs:
    assert path.is_file(), path
    print(
        path.name,
        f"{path.stat().st_size / 1024**2:.2f} MiB",
    )

subprocess.run(
    ["git", "-C", str(REPO), "diff", "--exit-code"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO), "diff", "--cached", "--exit-code"],
    check=True,
)

print()
print("BRATS RUN 1 PART 2 OUTPUT VERIFICATION: PASS")
print("Save this Kaggle notebook as a Version with outputs.")
print("Do not start a new run. Part 3 will resume at epoch 51.")


EMCAD_BraTS_Run1_Part2_State.tar.gz 380.07 MiB
EMCAD_BraTS_Run1_Part2_SHA256SUMS.txt 0.00 MiB
EMCAD_BraTS_Run1_Part2_stdout.log 0.01 MiB

BRATS RUN 1 PART 2 OUTPUT VERIFICATION: PASS
Save this Kaggle notebook as a Version with outputs.
Do not start a new run. Part 3 will resume at epoch 51.
